# Simulated patient heart model generation

This notebook generates *simulated* per-patient heart meshes by scaling each segmented component of the healthy heart mesh according to clinical conditions.

Inputs:
- `data-processing/cropped_data/hvsmr_clinical.csv`
- `models/condition_effects.json`
- `heart_models/healthy_model.obj` (segmented by `g` groups)

Outputs:
- `heart_models/simulated_patient_models/sim_healthy_to_patXX.obj`


In [ ]:
# Imports + paths
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data-processing'

clinical_path = DATA_DIR / 'hvsmr_clinical.csv'
if not clinical_path.exists():
    raise FileNotFoundError(f'Missing: {clinical_path}')

hvsmr_clinical = pd.read_csv(clinical_path)

condition_json_path = PROJECT_ROOT / 'models' / 'condition_effects.json'
if not condition_json_path.exists():
    raise FileNotFoundError(f'Missing: {condition_json_path}')

with open(condition_json_path, 'r', encoding='utf-8') as f:
    condition_effects = json.load(f)

cond_mult = condition_effects['condition_multipliers']

heart_models_dir = PROJECT_ROOT / 'heart_models'
healthy_segmented_obj = heart_models_dir / 'healthy.obj'
if not healthy_segmented_obj.exists():
    raise FileNotFoundError(f'Missing: {healthy_segmented_obj}')

out_dir = heart_models_dir / 'simulated_patient_models'
out_dir.mkdir(parents=True, exist_ok=True)

print('Loaded clinical rows:', hvsmr_clinical.shape)
print('Healthy segmented OBJ:', healthy_segmented_obj)
print('Output dir:', out_dir)


In [ ]:
# Frontend-equivalent mapping + helpers

# From heart-scene.js
PART_NAME_TO_VOL_KEY = {
    'LV': 'Label_1_vol_ml',
    'RV': 'Label_2_vol_ml',
    'LA': 'Label_3_vol_ml',
    'RA': 'Label_4_vol_ml',
    'AO': 'Label_5_vol_ml',
    'PA': 'Label_6_vol_ml',
    'PV': 'Label_7_vol_ml',
    'SVC': 'Label_8_vol_ml',
}

# Map OBJ group names -> these part codes (aliases may differ by mesh authoring)
OBJ_GROUP_TO_PART = {
    'LV': 'LV',
    'RV': 'RV',
    'LA': 'LA',
    'RA': 'RA',
    'AO': 'AO',
    'Aorta': 'AO',
    'PA': 'PA',
    'PV': 'PV',
    'SVC': 'SVC',
    'IVC': None,  # no condition multiplier key
}

CORE_COLS = {'Pat', 'Age', 'Category'}
condition_cols = [c for c in hvsmr_clinical.columns if c not in CORE_COLS]


def patient_conditions(pat_id: int):
    row = hvsmr_clinical.loc[hvsmr_clinical['Pat'].astype(int) == int(pat_id)]
    if row.empty:
        raise KeyError(f'Patient {pat_id} not found in hvsmr_clinical')
    row = row.iloc[0]
    conds = []
    for c in condition_cols:
        v = row[c]
        if isinstance(v, str) and v.strip().upper() == 'X':
            conds.append(c)
    return conds


def compute_scales_for_conditions(selected_conditions, default=1.0):
    # Match heart-scene.js computeScalesForConditions:
    # - geometric mean of multipliers per part across conditions
    # - clamp multiplier to [0.2, 5]
    # - convert volume ratio to linear scale via cube root
    if not selected_conditions:
        return {part: float(default) for part in PART_NAME_TO_VOL_KEY.keys()}

    selected_conditions = [c for c in selected_conditions if c != 'Normal']
    scales = {part: float(default) for part in PART_NAME_TO_VOL_KEY.keys()}
    if not selected_conditions:
        return scales

    for part, vol_key in PART_NAME_TO_VOL_KEY.items():
        product = 1.0
        n = 0
        for cond in selected_conditions:
            mult = cond_mult.get(cond, {}).get(vol_key, None)
            if mult is not None and mult > 0:
                product *= float(mult)
                n += 1
        mult = (product ** (1 / n)) if n > 0 else 1.0
        mult = max(0.2, min(5.0, mult))
        scales[part] = float(np.cbrt(mult))
    return scales


In [ ]:
# OBJ parsing/writing utilities (grouped by `g`)

def parse_obj_group_faces(obj_path: Path):
    verts = []
    groups = {}
    group_order = []
    current_group = '__ungrouped__'

    def ensure_group(g):
        if g not in groups:
            groups[g] = []
            group_order.append(g)

    ensure_group(current_group)

    with open(obj_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if not line.strip() or line.startswith('#'):
                continue
            if line.startswith('v '):
                _, x, y, z, *rest = line.split()
                verts.append([float(x), float(y), float(z)])
            elif line.startswith('g '):
                current_group = line[2:].strip() or '__unnamed__'
                ensure_group(current_group)
            elif line.startswith('f '):
                parts = line.split()[1:]
                idxs = [int(p.split('/')[0]) - 1 for p in parts]
                if len(idxs) == 3:
                    groups[current_group].append(tuple(idxs))
                elif len(idxs) > 3:
                    for k in range(1, len(idxs) - 1):
                        groups[current_group].append((idxs[0], idxs[k], idxs[k + 1]))

    return np.asarray(verts, dtype=np.float32), groups, group_order


def scale_vertices_by_group(verts: np.ndarray, groups: dict, group_scales: dict):
    v_out = verts.copy()
    for gname, faces in groups.items():
        s = float(group_scales.get(gname, 1.0))
        if abs(s - 1.0) < 1e-8:
            continue
        idx = np.unique(np.asarray(faces, dtype=np.int64).ravel())
        if idx.size == 0:
            continue
        centroid = v_out[idx].mean(axis=0)
        v_out[idx] = centroid + (v_out[idx] - centroid) * s
    return v_out


def write_obj_grouped(out_path: Path, verts: np.ndarray, groups: dict, group_order):
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write('# Simulated patient mesh generated from healthy_model.obj\n')
        for v in verts:
            f.write(f'v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n')
        for g in group_order:
            f.write(f'g {g}\n')
            for (a, b, c) in groups.get(g, []):
                f.write(f'f {a+1} {b+1} {c+1}\n')


In [ ]:
# Generate simulated meshes for selected patients

healthy_verts, healthy_groups, group_order = parse_obj_group_faces(healthy_segmented_obj)
print('Parsed verts:', healthy_verts.shape)
print('Groups:', [g for g in group_order if g in healthy_groups])

# Generate for all patients by default
SIM_PAT_IDS = sorted(hvsmr_clinical['Pat'].dropna().astype(int).unique().tolist())

rows = []
for pat_id in SIM_PAT_IDS:
    conds = patient_conditions(pat_id)
    part_scales = compute_scales_for_conditions(conds)

    group_scales = {}
    for gname in healthy_groups.keys():
        part = OBJ_GROUP_TO_PART.get(gname, None)
        group_scales[gname] = float(part_scales.get(part, 1.0)) if part is not None else 1.0

    v_sim = scale_vertices_by_group(healthy_verts, healthy_groups, group_scales)
    out_path = out_dir / f'simulated_patient{pat_id}.obj'
    write_obj_grouped(out_path, v_sim, healthy_groups, group_order)

    row = {
        'pat': int(pat_id),
        'n_conditions': len(conds),
        'conditions': ','.join(conds),
        'out_path': str(out_path),
    }
    for g in group_order:
        if g in healthy_groups and g != '__ungrouped__':
            row[f'scale_{g}'] = group_scales.get(g, 1.0)
    rows.append(row)

sim_summary = pd.DataFrame(rows)
display(sim_summary.head(10))
print('Wrote', len(sim_summary), 'OBJ(s) to:', out_dir)
